# MultiTask Feedback: Imb_mLaBSE

In [ ]:
# ================== IMPROVED MULTITASK LABSE (IMBALANCE-AWARE) ==================
# Model: LaBSE Multi-task (Emotion: multilabel, Sentiment & Aspect: multiclass)

import os
import math
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, precision_recall_fscore_support
)
from sklearn.model_selection import KFold

# =================== BASELINES ===================
# # BERT / mBERT (bert-base multilingual)
# from transformers import BertTokenizer, BertModel
# tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
# tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')

# # DistilBertModel
# from transformers import DistilBertTokenizer, DistilBertModel
# tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

# # DistilBertModel-Multilingual
# from transformers import DistilBertTokenizer, DistilBertModel
# tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-multilingual-cased")

# # RemBERT
# from transformers import AutoTokenizer, AutoModel
# tokenizer = AutoTokenizer.from_pretrained("google/rembert")

# LaBSE
from transformers import AutoTokenizer, AutoModel
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/LaBSE")

# # XML-Roberta
# from transformers import XLMRobertaTokenizer, XLMRobertaModel
# tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

# ================== CONFIG ==================
CHECKPOINT_DIR = "outputModel/LaBSE/Imb-LaBSE"
BEST_NAME_MODEL = "best_model_multi_task_emoji_attention_fold"
TRAIN_CSV_PATH = "dataset/4. newlabel/extend/train.csv"
TEST_CSV_PATH  = "dataset/4. newlabel/extend/test.csv"
OUTPUT_PRED_BEST = "outputPrediksi/LaBSE/Imb-LaBSE/pred_feedbackEmo.csv"
OUTPUT_THRESHOLDS = "outputPrediksi/LaBSE/Imb-LaBSE/opt_thresholds.json"       # file 1
OUTPUT_TEMPERATURE = "outputPrediksi/LaBSE/Imb-LaBSE/opt_temperature.json"     # file 2 (baru)

MAX_LEN = 256
BATCH_SIZE = 8
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCH = 10
DROPOUT = 0.1
LR = 326347317195311e-05
KFOLD_SPLITS = 10
THRESHOLD = 0.5

EMOTION_LABELS = ['anger','anticipation','disgust','fear','joy','sadness', 'surprise', 'trust']
NUM_EMOTIONS = len(EMOTION_LABELS)
NUM_SENTIMENT = 3
NUM_ASPECT = 5

# ================== MODEL ==================
class MultiTask(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = AutoModel.from_pretrained("sentence-transformers/LaBSE")
        # self.backbone = BertModel.from_pretrained('bert-base-uncased')
        # self.backbone = DistilBertModel.from_pretrained("distilbert-base-uncased")
        # self.backbone = XLMRobertaModel.from_pretrained('xlm-roberta-base')
        self.dropout = nn.Dropout(DROPOUT)
        hidden_size = self.backbone.config.hidden_size
        self.classifier_emotion = nn.Linear(hidden_size, NUM_EMOTIONS)
        self.classifier_sentiment = nn.Linear(hidden_size, NUM_SENTIMENT)
        self.classifier_aspect = nn.Linear(hidden_size, NUM_ASPECT)

    def forward(self, input_ids, attention_mask):
        output = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled = output.last_hidden_state[:, 0]
        x = self.dropout(pooled)
        return (
            self.classifier_emotion(x),
            self.classifier_sentiment(x),
            self.classifier_aspect(x)
        )

# ================== DATA ==================
class MultiTaskDataset(Dataset):
    def __init__(self, df):
        self.texts = df['feedback'].tolist()
        self.emotions = df[EMOTION_LABELS].values.astype(float)
        self.sentiments = df['sentiment'].astype(int).values
        self.aspects = df['aspect'].astype(int).values

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = tokenizer(text, truncation=True, padding='max_length', max_length=MAX_LEN, return_tensors='pt')
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'emotion_labels': torch.tensor(self.emotions[idx], dtype=torch.float),
            'sentiment_label': torch.tensor(self.sentiments[idx], dtype=torch.long),
            'aspect_label': torch.tensor(self.aspects[idx], dtype=torch.long)
        }

# ================== IMBALANCE HANDLING ==================
def pos_weight_from_binary_counts(pos_counts, total):
    pos = np.clip(pos_counts.astype(float), 1.0, None)
    neg = np.maximum(total - pos, 1.0)
    return torch.tensor(neg / pos, dtype=torch.float)

def class_weights_from_counts(counts, num_classes):
    counts = np.clip(counts.astype(float), 1.0, None)
    inv = 1.0 / counts
    inv *= (num_classes / inv.sum())
    return torch.tensor(inv, dtype=torch.float)

def collect_counts_from_subset(subset):
    ds = subset.dataset
    idxs = subset.indices
    emo_mat = ds.emotions[idxs, :] if isinstance(idxs, np.ndarray) else np.array([ds.emotions[i] for i in idxs])
    emo_pos_counts = emo_mat.sum(axis=0)
    total_samples = emo_mat.shape[0]
    sen_arr = ds.sentiments[idxs] if isinstance(idxs, np.ndarray) else np.array([ds.sentiments[i] for i in idxs])
    asp_arr = ds.aspects[idxs] if isinstance(idxs, np.ndarray) else np.array([ds.aspects[i] for i in idxs])
    sen_counts = np.bincount(sen_arr, minlength=NUM_SENTIMENT)
    asp_counts = np.bincount(asp_arr, minlength=NUM_ASPECT)
    return emo_pos_counts, total_samples, sen_counts, asp_counts

# ================== TRAINING ==================
def train_one_fold(fold, model, dataset, kf_indices):
    train_idx, val_idx = kf_indices[fold]
    train_ds = Subset(dataset, train_idx)
    val_ds = Subset(dataset, val_idx)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

    emo_pos_counts, total, sen_counts, asp_counts = collect_counts_from_subset(train_ds)
    emo_pos_weight = pos_weight_from_binary_counts(emo_pos_counts, total).to(DEVICE)
    sen_weights = class_weights_from_counts(sen_counts, NUM_SENTIMENT).to(DEVICE)
    asp_weights = class_weights_from_counts(asp_counts, NUM_ASPECT).to(DEVICE)

    loss_fn_emotion = nn.BCEWithLogitsLoss(pos_weight=emo_pos_weight)
    loss_fn_sentiment = nn.CrossEntropyLoss(weight=sen_weights)
    loss_fn_aspect = nn.CrossEntropyLoss(weight=asp_weights)
    optimizer = AdamW(model.parameters(), lr=LR)
    best_val_loss = float('inf')

    for epoch in range(1, EPOCH + 1):
        model.train()
        for batch in tqdm(train_loader, desc=f"Fold {fold+1} - Epoch {epoch} - Training"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            emotion_labels = batch['emotion_labels'].to(DEVICE)
            sentiment_label = batch['sentiment_label'].to(DEVICE)
            aspect_label = batch['aspect_label'].to(DEVICE)

            le, ls, la = model(input_ids, attention_mask)
            loss = (
                loss_fn_emotion(le, emotion_labels) +
                loss_fn_sentiment(ls, sentiment_label) +
                loss_fn_aspect(la, aspect_label)
            )
            optimizer.zero_grad(); loss.backward(); optimizer.step()

        model.eval(); total_val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                emotion_labels = batch['emotion_labels'].to(DEVICE)
                sentiment_label = batch['sentiment_label'].to(DEVICE)
                aspect_label = batch['aspect_label'].to(DEVICE)
                le, ls, la = model(input_ids, attention_mask)
                loss = (
                    loss_fn_emotion(le, emotion_labels) +
                    loss_fn_sentiment(ls, sentiment_label) +
                    loss_fn_aspect(la, aspect_label)
                )
                total_val_loss += loss.item()
        avg_val_loss = total_val_loss / len(val_loader)
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, f"{BEST_NAME_MODEL}_fold{fold+1}.pt"))

# ================== K-FOLD DRIVER ==================
def run_kfold_training():
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    df = pd.read_csv(TRAIN_CSV_PATH)
    dataset = MultiTaskDataset(df)
    kf = KFold(n_splits=KFOLD_SPLITS, shuffle=True, random_state=42)
    indices = list(kf.split(np.arange(len(dataset))))
    for fold in range(KFOLD_SPLITS):
        print(f"\n========== Fold {fold+1}/{KFOLD_SPLITS} ==========")
        model = MultiTask().to(DEVICE)
        train_one_fold(fold, model, dataset, indices)

# ================== THRESHOLD TUNING ==================
def tune_thresholds_per_label(y_true, y_prob):
    thresholds = []
    grid = np.linspace(0.05, 0.95, 37)
    for i in range(y_true.shape[1]):
        yt, yp = y_true[:, i], y_prob[:, i]
        best_f1, best_t = 0, 0.5
        for t in grid:
            f1 = f1_score(yt, (yp >= t).astype(int), zero_division=0)
            if f1 > best_f1: best_f1, best_t = f1, t
        thresholds.append(best_t)
    return thresholds

# ================== ENSEMBLE INFERENCE ==================
def predict_ensemble():
    df = pd.read_csv(TEST_CSV_PATH)
    y_true_emotion = df[EMOTION_LABELS].values.astype(int)
    y_true_sentiment = df['sentiment'].astype(int).values
    y_true_aspect = df['aspect'].astype(int).values

    probs_emotion, logits_sentiment, logits_aspect = [], [], []
    with torch.no_grad():
        for fold in range(KFOLD_SPLITS):
            print(f"Inferencing Fold {fold+1}")
            model = MultiTask().to(DEVICE)
            path = os.path.join(CHECKPOINT_DIR, f"{BEST_NAME_MODEL}_fold{fold+1}.pt")
            model.load_state_dict(torch.load(path, map_location=DEVICE))
            model.eval()

            fold_probs, fold_sen, fold_asp = [], [], []
            for i in range(0, len(df), BATCH_SIZE):
                texts = df['feedback'].iloc[i:i+BATCH_SIZE].tolist()
                enc = tokenizer(texts, return_tensors='pt', padding=True, truncation=True, max_length=MAX_LEN)
                input_ids = enc['input_ids'].to(DEVICE)
                attention_mask = enc['attention_mask'].to(DEVICE)
                le, ls, la = model(input_ids, attention_mask)
                fold_probs.append(torch.sigmoid(le).cpu().numpy())
                fold_sen.append(ls.cpu().numpy())
                fold_asp.append(la.cpu().numpy())

            probs_emotion.append(np.vstack(fold_probs))
            logits_sentiment.append(np.vstack(fold_sen))
            logits_aspect.append(np.vstack(fold_asp))

    avg_probs_emotion = np.mean(probs_emotion, axis=0)
    avg_logits_sen = np.mean(logits_sentiment, axis=0)
    avg_logits_asp = np.mean(logits_aspect, axis=0)
    pred_sentiment = np.argmax(avg_logits_sen, axis=1)
    pred_aspect = np.argmax(avg_logits_asp, axis=1)

    thresholds = tune_thresholds_per_label(y_true_emotion, avg_probs_emotion)
    pred_emotion = (avg_probs_emotion >= np.array(thresholds)[None, :]).astype(int)

    evaluate_multitask(y_true_emotion, avg_probs_emotion, pred_emotion, y_true_sentiment, pred_sentiment, y_true_aspect, pred_aspect)

    out_df = df.copy()
    for i, lab in enumerate(EMOTION_LABELS):
        out_df[f"pred_{lab}"] = pred_emotion[:, i]
    out_df['pred_sentiment'] = pred_sentiment
    out_df['pred_aspect'] = pred_aspect
    os.makedirs(os.path.dirname(OUTPUT_PRED_BEST), exist_ok=True)
    out_df.to_csv(OUTPUT_PRED_BEST, index=False)
    print(f"Saved predictions to {OUTPUT_PRED_BEST}")

# ================== METRICS ==================
def evaluate_multitask(y_true_emotion, y_prob_emotion, y_pred_emotion, y_true_sent, y_pred_sent, y_true_asp, y_pred_asp):
    print("\n===== EMOTION =====")
    print_metrics(y_true_emotion, y_pred_emotion, y_prob_emotion, EMOTION_LABELS)

    acc = accuracy_score(y_true_sent, y_pred_sent)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true_sent, y_pred_sent, average='macro')
    print("\n===== SENTIMENT =====")
    print(f"Acc: {acc:.4f}, Prec: {prec:.4f}, Rec: {rec:.4f}, F1: {f1:.4f}")

    acc = accuracy_score(y_true_asp, y_pred_asp)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true_asp, y_pred_asp, average='macro')
    print("\n===== ASPECT =====")
    print(f"Acc: {acc:.4f}, Prec: {prec:.4f}, Rec: {rec:.4f}, F1: {f1:.4f}")

def print_metrics(y_true, y_pred, y_prob, label_names):
    # Subset accuracy: semua label benar sekaligus
    subset_acc = accuracy_score(y_true, y_pred)

    # Sample accuracy: proporsi benar per sample
    sample_acc = np.mean((y_true == y_pred).sum(axis=1) / y_true.shape[1])

    # Global (micro/macro)
    micro_prec = precision_score(y_true, y_pred, average='micro', zero_division=0)
    micro_rec  = recall_score(y_true, y_pred, average='micro', zero_division=0)
    micro_f1   = f1_score(y_true, y_pred, average='micro', zero_division=0)
    macro_prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    macro_rec  = recall_score(y_true, y_pred, average='macro', zero_division=0)
    macro_f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)

    # Per-label
    per_label_acc = (y_true == y_pred).mean(axis=0)
    per_label_prec = precision_score(y_true, y_pred, average=None, zero_division=0)
    per_label_rec  = recall_score(y_true, y_pred, average=None, zero_division=0)
    per_label_f1   = f1_score(y_true, y_pred, average=None, zero_division=0)
    per_label_auc = []
    for j in range(y_true.shape[1]):
        if len(np.unique(y_true[:, j])) == 2:
            try:
                auc = roc_auc_score(y_true[:, j], y_prob[:, j])
            except ValueError:
                auc = np.nan
        else:
            auc = np.nan
        per_label_auc.append(auc)
    macro_auc = np.nanmean(per_label_auc)

    # Print hasil
    print("\n===== EMOTION EVALUATION METRICS =====")
    print(f"Subset Accuracy   : {subset_acc:.4f}")
    print(f"Sample Accuracy   : {sample_acc:.4f}")
    print(f"Micro Precision   : {micro_prec:.4f}")
    print(f"Micro Recall      : {micro_rec:.4f}")
    print(f"Micro F1          : {micro_f1:.4f}")
    print(f"Macro Precision   : {macro_prec:.4f}")
    print(f"Macro Recall      : {macro_rec:.4f}")
    print(f"Macro F1          : {macro_f1:.4f}")
    print(f"Macro ROC-AUC     : {macro_auc:.4f}")

    print("\nPer-Label Metrics:")
    for i, name in enumerate(label_names):
        auc_val = per_label_auc[i]
        auc_str = f"{auc_val:.4f}" if auc_val == auc_val else "NaN"
        print(f"- {name:22s} | Acc: {per_label_acc[i]:.4f} "
              f"| P: {per_label_prec[i]:.4f} "
              f"| R: {per_label_rec[i]:.4f} "
              f"| F1: {per_label_f1[i]:.4f} "
              f"| AUC: {auc_str}")

# def print_metrics(y_true, y_pred, y_prob, label_names):
#     for i, lab in enumerate(label_names):
#         auc = roc_auc_score(y_true[:, i], y_prob[:, i]) if len(np.unique(y_true[:, i])) == 2 else float('nan')
#         print(f"{lab:15s} | P: {precision_score(y_true[:, i], y_pred[:, i], zero_division=0):.4f} "
#               f"R: {recall_score(y_true[:, i], y_pred[:, i], zero_division=0):.4f} "
#               f"F1: {f1_score(y_true[:, i], y_pred[:, i], zero_division=0):.4f} "
#               f"AUC: {auc:.4f}")

# ================== ENTRY ==================
if __name__ == "__main__":
    run_kfold_training()
    print("\n[ENSEMBLE EVALUATION]")
    predict_ensemble()
